<a href="https://colab.research.google.com/github/rodrigueshenri-sketch/banco-de-dados-3ds/blob/main/C%C3%B3pia_de_3ds_Analise_exploratoria_dados_faturamento.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 **ANÁLISE EXPLORATÓRIA DE DADOS DE FATURAMENTO**  

⚠️ATENÇÃO!!!⚠️ FAÇA UMA CÓPIA DESTE ARQUIVO NO SEU GOOGLE DRIVE ANTES DE COMEÇAR.  


 Clique no ícone de **Pasta** na lateral esquerda do Colab  
 e faça upload do arquivo **12-03 - Dashboard-base.xlsx**.

**Célula 1: Importação das Bibliotecas**

Nesta etapa, carregamos o pandas para tratar os dados e o plotly para criar gráficos interativos e modernos.

In [1]:
# codigo #

# Garante que o motor de leitura de Excel esteja atualizado

!pip install --upgrade openpyxl

# Importa bibliotecas necessárias do Python
import pandas as pd
import plotly.express as px
import plotly.io as pio

# Configura o gráfico para aparecer no Google Colab
pio.renderers.default = "colab"
print("✅ Bibliotecas prontas!")



✅ Bibliotecas prontas!


**Célula 2: Carregamento da Base de Dados e Amostra**  

*Nota: Certifique-se de que o arquivo "12-03 - Dashboard-base.xlsx" foi carregado na pasta lateral do Colab.*




In [2]:
# codigo #
# Lendo a aba específica chamada 'Base'
nome_arquivo = '12-03 - Dashboard-base.xlsx'
df = pd.read_excel(nome_arquivo, sheet_name='Base')

print("Amostra dos Dados (Primeiras 5 linhas):")
df.head()


Amostra dos Dados (Primeiras 5 linhas):


/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning:

Unknown extension is not supported and will be removed



,Data,Ano,Mês,Vendedor,Cliente,Região,Produto,Valor
0,2019-01-01,2019,jan,Paulo,Thamires Bastos,Sudeste,Python,450
1,2019-01-01,2019,jan,Paulo,Jessika Mineiro,Centro-Oeste,Excel,500
2,2019-01-02,2019,jan,Diego,Glenda Jalles,Norte,VBA,600
3,2019-01-02,2019,jan,Diego,Eugênio Mattos,Sul,Python,450
4,2019-01-02,2019,jan,Alon,Yasmine Gomes,Norte,VBA,600


**Célula 3: Estrutura da Base de Dados**

Aqui verificamos os tipos de colunas e se há necessidade de ajustes (como converter a Data).

In [3]:
# código #
# Mostra a estrutura da base de dados

print("--- Informações da Estrutura ---")
print(df.info())

# Garantir que a coluna Data seja do tipo datetime
df['Data'] = pd.to_datetime(df['Data'])

print("\nValores nulos por coluna:")
print(df.isnull().sum())


--- Informações da Estrutura ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3669 entries, 0 to 3668
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   Data      3669 non-null   datetime64[ns]
 1   Ano       3669 non-null   int64         
 2   Mês       3669 non-null   object        
 3   Vendedor  3669 non-null   object        
 4   Cliente   3669 non-null   object        
 5   Região    3669 non-null   object        
 6   Produto   3669 non-null   object        
 7   Valor     3669 non-null   int64         
dtypes: datetime64[ns](1), int64(2), object(5)
memory usage: 229.4+ KB
None

Valores nulos por coluna:
Data        0
Ano         0
Mês         0
Vendedor    0
Cliente     0
Região      0
Produto     0
Valor       0
dtype: int64


**Célula 4: Resumo Estatístico**

Uma visão geral sobre os valores financeiros da planilha.

In [4]:
#  codigo  #
# Resumo estatístico do faturamento (valor)
resumo = df['Valor'].describe().to_frame()
resumo.columns = ['Estatísticas de Faturamento']
resumo


,Estatísticas de Faturamento
count,3669.000000
mean,462.264922
std,108.207461
min,300.000000
25%,300.000000
50%,450.000000
75%,500.000000
max,600.000000


**Célula 5: Gráfico 1 - Faturamento Total por Vendedor**

Este gráfico de barras horizontais ajuda a identificar rapidamente quem são os melhores vendedores, similar ao dashboard original.

In [5]:
#  codigo  #
# Gráfico de Faturamento por Vendedor
vendas_vendedor = df.groupby('Vendedor')['Valor'].sum().reset_index().sort_values('Valor', ascending=True)

fig1 = px.bar(vendas_vendedor,
             x='Valor',
             y='Vendedor',
             orientation='h',
             title='Faturamento Total por Vendedor',
             text_auto='.2s',
             color='Valor',
             color_continuous_scale='Viridis')

fig1.update_layout(yaxis={'categoryorder':'total ascending'})
fig1.show()



**Célula 6: Gráfico 2 - Distribuição por Região**

Um gráfico de rosca para entender a participação de cada mercado.

In [6]:
#  codigo  #
# Gráfico do Faturamento por Região
vendas_regiao = df.groupby('Região')['Valor'].sum().reset_index()

fig2 = px.pie(vendas_regiao,
             values='Valor',
             names='Região',
             title='Participação do Faturamento por Região',
             hole=0.5,
             color_discrete_sequence=px.colors.qualitative.Pastel)

fig2.update_traces(textinfo='percent+label')
fig2.show()



**Célula 7: Gráfico 3 - Evolução Mensal do Faturamento**

Gráfico de linhas para visualizar a sazonalidade e o crescimento ao longo dos meses.

In [8]:
#   codigo  #


# Gráfico comparativo de faturamentos - 2019 a 2021

# Ordenação dos meses para o gráfico ficar cronológico
ordem_meses = ['jan', 'fev', 'mar', 'abr', 'mai', 'jun', 'jul', 'ago', 'set', 'out', 'nov', 'dez']

# Agrupamento por Ano e Mês
df_evolucao = df.groupby(['Ano', 'Mês'])['Valor'].sum().reset_index()
df_evolucao['Mês'] = pd.Categorical(df_evolucao['Mês'], categories=ordem_meses, ordered=True)
df_evolucao = df_evolucao.sort_values(['Ano', 'Mês'])

fig3 = px.line(df_evolucao,
              x='Mês',
              y='Valor',
              color='Ano',
              title='Evolução Mensal do Faturamento (Comparativo Anual)',
              markers=True,
              line_shape='linear')

fig3.show()

**Célula 8: Gráfico 4 - Faturamento por Produto e Região**

Gráfico de histogramas para visualizar e comparar o faturamento por produto em cada região do país.

In [9]:
#   codigo #


# Gráfico de Faturamento por Produto e Região

px.histogram(df, x='Produto', y='Valor', color='Região', barmode='group', title='Faturamento por Produto e Região')


***😃A análise exploratória de dados está concluída!***